# 1. Instalaciones

In [3]:
%pip install azure-search-documents==11.4.0 openai==1.3.0 dotenv python-dateutil

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
%pip uninstall httpx -y
%pip install httpx==0.26.0


Found existing installation: httpx 0.26.0
Uninstalling httpx-0.26.0:
  Successfully uninstalled httpx-0.26.0
Note: you may need to restart the kernel to use updated packages.
^C
Note: you may need to restart the kernel to use updated packages.


  Using cached httpx-0.26.0-py3-none-any.whl.metadata (7.6 kB)
Using cached httpx-0.26.0-py3-none-any.whl (75 kB)



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# 2. Importaciones

In [4]:
import os
import re
import uuid
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from openai import AzureOpenAI
from dotenv import load_dotenv
import json
import os
import json
import uuid
from typing import List, Dict, Any
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.storage.blob import BlobServiceClient
from openai import AzureOpenAI
from dotenv import load_dotenv
from datetime import datetime
import time
import traceback

# 3. Configuración

In [18]:
# Cargar variables del entorno
load_dotenv()

# Azure Search
search_endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
search_key = os.getenv("AZURE_SEARCH_KEY")
index_name = "ponal-documents-index"

# Azure OpenAI
openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
openai_key = os.getenv("AZURE_OPENAI_API_KEY")

# Azure Blob Storage
blob_connection_string = os.getenv("BLOB_CONNECTION_STRING")
container_name = os.getenv("BRONZE_CONTAINER_NAME")
blob_prefix = "servicio_policia/servicio_policia_processed/"

# Verificar configuraciones
def verificar_configuracion():
    required = {
        "AZURE_SEARCH_ENDPOINT": search_endpoint,
        "AZURE_SEARCH_KEY": search_key,
        "AZURE_OPENAI_ENDPOINT": openai_endpoint,
        "AZURE_OPENAI_API_KEY": openai_key,
        "BLOB_CONNECTION_STRING": blob_connection_string
    }
    
    missing = [name for name, value in required.items() if not value]
    if missing:
        print(f"❌ Variables faltantes en .env: {', '.join(missing)}")
        return False
    
    print("✅ Configuración verificada correctamente")
    return True

if not verificar_configuracion():
    raise ValueError("Configuración incompleta")

# Cliente Azure Search
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=index_name,
    credential=AzureKeyCredential(search_key)
)

# Cliente Azure OpenAI
openai_client = AzureOpenAI(
    azure_endpoint=openai_endpoint,
    api_key=openai_key,
    api_version="2024-02-01"
)

# Cliente Blob Storage
blob_service_client = BlobServiceClient.from_connection_string(blob_connection_string)
container_client = blob_service_client.get_container_client(container_name)

print("✅ Clientes inicializados")

✅ Configuración verificada correctamente
✅ Clientes inicializados


# 4. Función para limpiar ID

In [19]:
def limpiar_id(id_texto: str) -> str:
    """Limpia un ID para que sea válido en Azure Search"""
    if not id_texto:
        return f"id_{uuid.uuid4().hex[:8]}"
    
    # Reemplazar espacios y caracteres especiales
    limpio = re.sub(r'[^a-zA-Z0-9_-]', '_', str(id_texto))
    
    # Asegurar que no empiece con guión
    limpio = limpio.lstrip('_-')
    
    # Si queda vacío, generar ID aleatorio
    if not limpio:
        limpio = f"id_{uuid.uuid4().hex[:8]}"
    
    return limpio

# 5. Función para generar embeddings

In [20]:
def generar_embedding(texto: str) -> List[float]:
    """Genera embedding usando Azure OpenAI"""
    try:
        # Limitar texto para evitar exceder límites
        texto_limitado = texto[:8000]
        
        print(f"  Generando embedding ({len(texto_limitado)} caracteres)...")

        response = openai_client.embeddings.create(
            input=texto_limitado,
            model="text-embedding-3-large"  # Modelo estándar
        )
        
        embedding = response.data[0].embedding
        print(f"  ✅ Embedding generado ({len(embedding)} dimensiones)")
        return embedding
        
    except Exception as e:
        print(f"  ❌ Error generando embedding: {str(e)[:100]}")
        return [0.0] * 3072

# 6. Función para preparar los documentos

In [21]:
def preparar_documento_para_indexacion(chunk_data: Dict[str, Any], blob_path: str) -> Dict[str, Any]:
    """Prepara un documento para ser indexado en Azure Search"""
    try:
        # Obtener IDs y limpiarlos
        doc_id_original = chunk_data.get("id", f"doc_{uuid.uuid4().hex[:8]}")
        chunk_id_original = chunk_data.get("chunk_id", "")
        
        doc_id = limpiar_id(doc_id_original)
        chunk_id = limpiar_id(chunk_id_original)
        
        # Crear ID único y limpio para Azure Search
        if chunk_id:
            search_id = f"{doc_id}_{chunk_id}"
        else:
            search_id = doc_id
        
        # Obtener texto del contenido
        texto = ""
        for campo in ["content", "text", "Content", "Text"]:
            if campo in chunk_data and chunk_data[campo]:
                texto = str(chunk_data[campo]).strip()
                break
        
        # Si no hay texto, usar una descripción mínima
        if not texto:
            texto = f"Documento {doc_id} - Chunk {chunk_id}"
        
        # Generar embedding
        embedding = generar_embedding(texto)
        
        # Obtener nombre de archivo
        nombre_archivo = chunk_data.get("filename", "")
        if not nombre_archivo:
            nombre_archivo = os.path.basename(blob_path)
            if nombre_archivo.endswith('.json'):
                nombre_archivo = nombre_archivo[:-5]
        
        # Construir documento final
        documento = {
            "id": search_id,  # ID limpio y válido
            "chunk_id": chunk_id,
            "content": texto,
            "filename": nombre_archivo,
            "filepath": blob_path,
            "file_type": chunk_data.get("file_type", "unknown"),
            "blob_url": f"https://{blob_service_client.account_name}.blob.core.windows.net/{container_name}/{blob_path}",
            "embedding": embedding
        }
        
        # Añadir campos opcionales si existen
        campos_opcionales = ["section", "pages", "pages_total", "page_start", 
                           "page_end", "chunk_index", "total_chunks"]
        
        for campo in campos_opcionales:
            if campo in chunk_data:
                documento[campo] = chunk_data[campo]
        
        print(f"  ✅ Documento preparado: {search_id}")
        return documento
        
    except Exception as e:
        print(f"  ❌ Error preparando documento: {str(e)[:100]}")
        return None

# 7. Función para procesar los documentos

In [31]:
def procesar_documentos_desde_blob() -> List[Dict[str, Any]]:
    """Procesa todos los archivos JSON del Blob Storage"""
    print(f"📂 Procesando documentos desde: {container_name}/{blob_prefix}")
    
    documentos_preparados = []
    archivos_procesados = 0
    errores = 0

    archivos = ('2DC-PR-0002 TRATAMIENTO Y ANÁLISIS DE LA EVIDENCIA DIGITAL.json',
                '2DC-PR-0017 BÚSQUEDA PROSPECCIÓN EXCAVACIÓN EXHUMACIÓN.json',
                '2DC-PR-0026 RECOLECTAR DATOS VOLÁTILES..json',
                '2DC-PR-0027 EXTRACCIÓN DE INFORMACIÓN A EQUIPOS TERMINALES MOVILES.json',
                '2DC-PR-0031 REALIZAR TOMA Y SISTEMATIZACIÓN DE CARTA DENTAL.json',
                '2DC-PR-0033 REALIZAR IMAGENES FORENSES.json',
                '2DC-PR-0035 ANÁLISIS BIOANTROPOLÓGICO, NECROPSIA MÉDICO LEGAL, IDENTIFICACIÓN, EMISIÓN DE D.json',
                '2DC-PR-0037 REALIZAR ANÁLISIS E INFORME ODONTOLÓGICO FORENSE.json',
                '2DI-GU-0002 GUÍA PARA EL EMPLEO DE LAS ESTADÍSTICA EN LAPOLICÍA NACIONAL.json',
                '2DI-PR-0002 ELABORAR Y PUBLICAR REVISTA CRIMINALIDAD.json',
                '2DI-PR-0004 RESPUESTA REQUERIMIENTOS ESTADÍSTICOS.json',
                '2EI-MA-0002 MANUAL PARA LA ERRADICACIÓN DE CULTIVOS ILÍCITOS.json',
                '2IJ-GU-0002 GUÍA PARA LA APLICACIÓN DE LA TÉCNICA EN PERFILACIÓN CRIMINAL.json',
                '2IJ-GU-0003 TRATAMIENTO Y DISPOSICIÓN FINAL DE ELEMENTOS MATERIALES PROBATORIOS Y EVIDENCIA .json',
                'Manual-de-Policia-Judicial-Actualizado.json')
    
    try:
        # Listar archivos JSON
        blobs = container_client.list_blobs(name_starts_with=blob_prefix)
        archivos_json = [
            #b for b in blobs if b.name.endswith(.json)]
            b for b in blobs if b.name.endswith(archivos)]
        
        print(f"📊 Encontrados {len(archivos_json)} archivos JSON")
        
        for i, blob in enumerate(archivos_json):
            try:
                print(f"\n📄 Procesando archivo {i+1}/{len(archivos_json)}: {blob.name}")
                
                # Descargar archivo
                blob_client = container_client.get_blob_client(blob.name)
                contenido = blob_client.download_blob().readall()
                
                # Parsear JSON
                chunks = json.loads(contenido)
                
                if not isinstance(chunks, list):
                    print(f"  ⚠️  El archivo no contiene una lista, omitiendo")
                    continue
                
                print(f"  📋 Encontrados {len(chunks)} chunks")
                
                # Procesar cada chunk
                chunks_procesados = 0
                for chunk in chunks:
                    documento = preparar_documento_para_indexacion(chunk, blob.name)
                    if documento:
                        documentos_preparados.append(documento)
                        chunks_procesados += 1
                
                print(f"  ✅ Chunks procesados: {chunks_procesados}")
                archivos_procesados += 1
                
            except Exception as e:
                print(f"  ❌ Error procesando archivo: {str(e)[:100]}")
                errores += 1
        
        print(f"\n{'='*60}")
        print(f"📊 RESUMEN DE PROCESAMIENTO")
        print(f"{'='*60}")
        print(f"Archivos procesados: {archivos_procesados}")
        print(f"Documentos preparados: {len(documentos_preparados)}")
        print(f"Errores: {errores}")
        
        if documentos_preparados:
            print(f"\n📄 Ejemplo de documento preparado:")
            ejemplo = documentos_preparados[0]
            print(f"  ID: {ejemplo.get('id')}")
            print(f"  Archivo: {ejemplo.get('filename')}")
            print(f"  Texto: {ejemplo.get('content', '')[:80]}...")
        
        return documentos_preparados
        
    except Exception as e:
        print(f"❌ Error general: {str(e)}")
        return []

# 7. Función para indexar

In [32]:
def indexar_documentos(documentos: List[Dict[str, Any]], tamano_lote: int = 50) -> Dict[str, Any]:
    """Indexa documentos en Azure Search"""
    if not documentos:
        return {"exitosos": 0, "fallidos": 0, "mensaje": "No hay documentos para indexar"}
    
    print(f"\n🚀 Iniciando indexación de {len(documentos)} documentos...")
    
    exitosos = 0
    fallidos = 0
    errores_detallados = []
    
    # Indexar en lotes
    for inicio in range(0, len(documentos), tamano_lote):
        fin = min(inicio + tamano_lote, len(documentos))
        lote = documentos[inicio:fin]
        num_lote = (inicio // tamano_lote) + 1
        
        print(f"\n📦 Lote {num_lote}: documentos {inicio+1}-{fin}")
        
        try:
            # Subir lote a Azure Search
            resultados = search_client.upload_documents(documents=lote)
            
            # Contar resultados
            exitosos_lote = sum(1 for r in resultados if r.succeeded)
            fallidos_lote = sum(1 for r in resultados if not r.succeeded)
            
            exitosos += exitosos_lote
            fallidos += fallidos_lote
            
            print(f"  ✅ Indexados: {exitosos_lote}")
            print(f"  ❌ Fallidos: {fallidos_lote}")
            
            # Registrar errores
            for r in resultados:
                if not r.succeeded:
                    errores_detallados.append({
                        "documento": r.key if hasattr(r, 'key') else "desconocido",
                        "error": r.error_message if hasattr(r, 'error_message') else str(r)
                    })
            
            # Pequeña pausa entre lotes
            if fin < len(documentos):
                time.sleep(0.5)
                
        except Exception as e:
            print(f"  ❌ Error en lote {num_lote}: {str(e)[:100]}")
            fallidos += len(lote)
            errores_detallados.append({"lote": num_lote, "error": str(e)})
    
    # Resultado final
    print(f"\n{'='*60}")
    print(f"📊 RESULTADO FINAL DE INDEXACIÓN")
    print(f"{'='*60}")
    print(f"Documentos exitosos: {exitosos}")
    print(f"Documentos fallidos: {fallidos}")
    
    if errores_detallados:
        print(f"\n🔍 Primeros 3 errores:")
        for i, error in enumerate(errores_detallados[:3]):
            print(f"  {i+1}. {error}")
    
    return {
        "exitosos": exitosos,
        "fallidos": fallidos,
        "total": len(documentos),
        "errores": errores_detallados[:5]  # Limitar a 5 errores
    }

# 8. Función para ejecutar indexación

In [33]:
def ejecutar_pipeline_completo():
    """Ejecuta todo el pipeline de indexación"""
    print(f"\n{'='*60}")
    print(f"🚀 INICIANDO PIPELINE DE INDEXACIÓN")
    print(f"{'='*60}")
    print(f"Hora inicio: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    inicio = time.time()
    
    try:
        # Paso 1: Procesar documentos del Blob Storage
        print(f"\n{'='*60}")
        print(f"📥 PASO 1: PROCESANDO DOCUMENTOS")
        print(f"{'='*60}")
        
        documentos = procesar_documentos_desde_blob()
        
        if not documentos:
            print("❌ No se encontraron documentos para indexar")
            return {"exito": False, "mensaje": "No hay documentos"}
        
        # Paso 2: Indexar en Azure Search
        print(f"\n{'='*60}")
        print(f"📤 PASO 2: INDEXANDO EN AZURE SEARCH")
        print(f"{'='*60}")
        
        resultado = indexar_documentos(documentos, tamano_lote=50)
        
        # Calcular tiempo
        fin = time.time()
        duracion = fin - inicio
        
        # Mostrar resumen final
        print(f"\n{'='*60}")
        print(f"🎉 PIPELINE COMPLETADO")
        print(f"{'='*60}")
        print(f"Hora fin: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"Duración: {duracion:.2f} segundos")
        print(f"Documentos procesados: {resultado['total']}")
        print(f"Documentos indexados: {resultado['exitosos']}")
        
        if resultado['exitosos'] > 0:
            print(f"\n✅ ¡Pipeline ejecutado con éxito!")
            return {"exito": True, **resultado}
        else:
            print(f"\n⚠️  Pipeline completado con errores")
            return {"exito": False, **resultado}
            
    except Exception as e:
        print(f"\n❌ Error en el pipeline: {str(e)}")
        return {"exito": False, "mensaje": str(e)}


# 9. Ejecutar indexación

In [34]:
print(f"\n{'='*60}")
print(f"▶️  EJECUTANDO PIPELINE DE INDEXACIÓN COMPLETO")
print(f"{'='*60}")

resultado = ejecutar_pipeline_completo()

if resultado.get("exito"):
    print(f"\n🎉 ¡Pipeline completado exitosamente!")
    print(f"   Documentos indexados: {resultado.get('exitosos', 0)}")
else:
    print(f"\n❌ Pipeline falló: {resultado.get('mensaje', 'Error desconocido')}")


▶️  EJECUTANDO PIPELINE DE INDEXACIÓN COMPLETO

🚀 INICIANDO PIPELINE DE INDEXACIÓN
Hora inicio: 2026-01-28 12:24:15

📥 PASO 1: PROCESANDO DOCUMENTOS
📂 Procesando documentos desde: bronze/servicio_policia/servicio_policia_processed/
📊 Encontrados 15 archivos JSON

📄 Procesando archivo 1/15: servicio_policia/servicio_policia_processed/2DC-PR-0002 TRATAMIENTO Y ANÁLISIS DE LA EVIDENCIA DIGITAL.json
  📋 Encontrados 10 chunks
  Generando embedding (953 caracteres)...
  ✅ Embedding generado (3072 dimensiones)
  ✅ Documento preparado: 5c24a3602cf10bf5b6519c2491a83eeb_2DC-PR-0002_TRATAMIENTO_Y_AN_LISIS_DE_LA_EVIDENCIA_DIGITAL_0
  Generando embedding (913 caracteres)...
  ✅ Embedding generado (3072 dimensiones)
  ✅ Documento preparado: a0f5a4372d6765188f163a176d1b45e9_2DC-PR-0002_TRATAMIENTO_Y_AN_LISIS_DE_LA_EVIDENCIA_DIGITAL_1
  Generando embedding (937 caracteres)...
  ✅ Embedding generado (3072 dimensiones)
  ✅ Documento preparado: 802b783a3b6edba908fa32fecbbc4246_2DC-PR-0002_TRATAMIENTO_Y_